<a href="https://colab.research.google.com/github/human720/Cek-Ketersediaan/blob/main/Cek_Ketersediaan_(uas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# VALIDASI WAKTU DAN JADWAL
# =========================
def validasi_input_jadwal(tanggal, mulai, selesai):
    try:
        tgl = datetime.strptime(tanggal, "%d-%m-%Y").date()
        jam_mulai = datetime.strptime(mulai, "%H:%M").time()
        jam_selesai = datetime.strptime(selesai, "%H:%M").time()
    except ValueError:
        return None, "Format tanggal/jam salah. Gunakan DD-MM-YYYY dan HH:MM."

    if tgl < date.today():
        return None, "Tanggal booking tidak boleh di masa lalu."

    if jam_mulai.minute != 0 or jam_selesai.minute != 0:
        return None, "Untuk program ini, pemesanan menggunakan kelipatan 1 jam."

    if jam_mulai.hour < JAM_BUKA or jam_selesai.hour > JAM_TUTUP:
        return None, "Jam operasional adalah 08:00 sampai 22:00."

    if jam_selesai <= jam_mulai:
        return None, "Jam selesai harus lebih besar dari jam mulai."

    durasi = (datetime.combine(tgl, jam_selesai) -
              datetime.combine(tgl, jam_mulai)).total_seconds() / 3600

    if durasi < 1:
        return None, "Durasi booking minimal 1 jam."

    return (tgl, jam_mulai, jam_selesai), None


def rekomendasi_slot(id_fasilitas, tanggal):
    print("\nRekomendasi slot kosong:")
    ditemukan = False

    for jam in range(JAM_BUKA, JAM_TUTUP):
        mulai = f"{jam:02d}:00"
        selesai = f"{jam + 1:02d}:00"
        if not cek_bentrok(id_fasilitas, tanggal, mulai, selesai):
            print(f"- {mulai} - {selesai}")
            ditemukan = True

    if not ditemukan:
        print("Tidak ada slot kosong pada tanggal tersebut.")


def cek_ketersediaan():
    lihat_fasilitas()
    try:
        id_fasilitas = int(input("\nID fasilitas : "))
    except ValueError:
        print("ID fasilitas harus berupa angka.")
        return

    tanggal = input("Tanggal (DD-MM-YYYY) : ").strip()
    mulai = input("Jam mulai (HH:MM)     : ").strip()
    selesai = input("Jam selesai (HH:MM)   : ").strip()

    hasil, pesan = validasi_input_jadwal(tanggal, mulai, selesai)
    if pesan:
        print(pesan)
        return

    with koneksi() as conn:
        cur = conn.cursor()
        cur.execute("SELECT id FROM fasilitas WHERE id = ?", (id_fasilitas,))
        if not cur.fetchone():
            print("Fasilitas tidak ditemukan.")
            return

    if cek_bentrok(id_fasilitas, tanggal, mulai, selesai):
        print("\nJadwal sudah digunakan. Silakan pilih jam lain.")
        rekomendasi_slot(id_fasilitas, tanggal)
    else:
        print("\nJadwal tersedia dan dapat dipesan.")